这个notebook主要作用：
1. 生成 video 训练数据 目录，即 `ANNO_FILE`
2. merge 不同 video 训练数据 目录， 可以shuffle/sort
3. merge 不同 音乐 目录pkl，从小pkl合并成大pkl

In [1]:
import sys
sys.path.append("/opt/tiger/VideoSiglip2-music") 
import pickle
import json
from dataset.hdfs_io import hlist_files, hisdir, hopen
import subprocess
import os
import random
from concurrent.futures import ThreadPoolExecutor

/usr/local/lib/python3.11/dist-packages/bytedmetrics/__init__.py:10: UserWarning: bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`
  warnings.warn("bytedmetrics is renamed to bytedance.metrics, please using `bytedance.metrics` instead of `bytedmetrics`")


Tokenizer_Pipeline is not installed, please install the matx first


## Merge video data

In [2]:
HADOOP_BIN = 'HADOOP_ROOT_LOGGER=ERROR,console hdfs'

def list_leaf_files_ls(root, skip_part=None):
    cmd = f"{HADOOP_BIN} dfs -ls -R {root}"
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    files = []
    for bline in p.stdout:
        line = bline.decode('utf8').strip()
        parts = line.split()
        if len(parts) >= 8 and parts[0].startswith('-'):
            if skip_part and f"{root}/{skip_part}" in parts[-1]:
                continue
            path = parts[-1]
            if '_SUCCESS' in path or '_temporary' in path or '.caption' in path:
                continue
            files.append(path)
    p.stdout.close(); p.wait()
    return sorted(files)

In [3]:
# root = "hdfs://harunava/user/wangxiuqi.0601/lutong/video/merged_train_data/20251020_1300w"
# root = "hdfs://harunava/user/wangxiuqi.0601/lutong/video/merged_train_data/20251017_2800w"
# root = "hdfs://harunava/user/wangxiuqi.0601/lutong/video/merged_train_data/20251106_2000w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20251017_1800w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20251020_1200w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20251106_2000w_filtered"
# root = "hdfs://harunava/user/wangxiuqi.0601/guyanhang/panel_capsule_in_60m_item" # subset from 50m, panel/capsule only
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260126_unpublish_2000w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260219_220w_all"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260219_220w_all_shuffled"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260219_220w_all_pairwise"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260310_1000w_all"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260323_2w2_all"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260327_5000w_highscore_730w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260327_300w_highscore_100w"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w"
# root = "hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w"
# root = "hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20251017_1800w"
# root = "hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20251020_1200w"
# root = "hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20251106_2000w_filtered"

# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260223_650w_all"
# root = "hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260301_82w_all"
root=" hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all"
# files = list_leaf_files_ls(root, skip_part="part-00080")
files = list_leaf_files_ls(root)
len(files) # 24300

200

In [4]:
files[0:5]

['hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all/part-00000-959f8386-758c-4226-b390-97e1e0a9403c-c000.json.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all/part-00001-959f8386-758c-4226-b390-97e1e0a9403c-c000.json.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all/part-00002-959f8386-758c-4226-b390-97e1e0a9403c-c000.json.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all/part-00003-959f8386-758c-4226-b390-97e1e0a9403c-c000.json.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260416_neg_300w_all/part-00004-959f8386-758c-4226-b390-97e1e0a9403c-c000.json.snappy']

In [4]:
files[0:5]

['hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00000-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00001-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00002-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00003-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunafr/home/byte_video_rec_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00004-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy']

In [4]:
files[0:5]

['hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00000-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00001-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00002-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00003-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy',
 'hdfs://harunava/home/byte_tiktok_music/proj/content_understanding/merged_train_data_v5/20260331_5000w_n_300w_high_5100w/part-00004-79cfd456-6c51-47c7-8e46-c3e6af57f435-c000.txt.snappy']

In [5]:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/llm_judge_pretrain_data_pgc_1106_19m.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260128_20m_unpublish.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260220_220w.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260220_220w_pairwise_880w.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260223_650w.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260301_82w.json', 'w') as fw:
#     json.dump(files, fw, indent=2)

# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260311_300w_highscore_1000w.json', 'w') as fw:
#     json.dump(files, fw, indent=2)


# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260323_80w_dedup_2w2.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260328_5000w_highscore_730w.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260328_300w_highscore_100w.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260331_5000w_n_300w_high_5100w_shuffled.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_v5_score_20260331_5000w_n_300w_high_5100w_shuffled.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1017_18m.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1020_12m.json', 'w') as fw:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1106_20m.json', 'w') as fw:
#     json.dump(files, fw, indent=2)
with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/va_train_data_pgc_v5_score_20260424_neg_300w.json', 'w') as fw:
    json.dump(files, fw, indent=2)

In [6]:
def concat_json_lists(in1, in2, out, dedup=True, sort_out=True, shuffle_out=False, seed=42):
    with open(in1, 'r') as f1, open(in2, 'r') as f2:
        a = json.load(f1)
        b = json.load(f2)
    print(len(a))
    print(len(b))
    merged = a + b
    if dedup:
        merged = list(dict.fromkeys(merged))  # preserves first-seen order
    if shuffle_out:
        rng = random.Random(seed) if seed is not None else random
        rng.shuffle(merged)
        print('shuffled')
    elif sort_out:
        merged = sorted(merged)
        print('sorted')
    print(len(merged))
    with open(out, 'w') as fo:
        json.dump(merged, fo, indent=2)

In [7]:
# prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/llm_judge_pretrain_data_pgc_1023_41m_new.json"
# new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/llm_judge_pretrain_data_pgc_1106_19m.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/llm_judge_pretrain_data_pgc_1106_60m.json"


# prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260116_30m.json"
# new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_1106_20m.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260116_50m.json"

# # prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_1106_10m_panel_capsule_only.json"
# prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260116_50m.json"
# new =  "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260128_20m_unpublish.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260128_70m_pub_n_unpub.json"


# prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_1106_10m_panel_capsule_only.json"
# new =  "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260128_70m_pub_n_unpub.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260128_80m_pub_n_unpub.json"

# prev = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260220_220w.json"
# new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260301_82w.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260301_302w.json"

# prev =       "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_pgc_v5_score_20260116_50m.json"
# new =        "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260301_302w.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260313_53m.json"

# prev =       "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260328_5000w_highscore_730w.json"
# new =        "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260328_300w_highscore_100w.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260328_highscore_830w.json"

# prev =       "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1017_18m.json"
# new =        "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1020_12m.json"
# merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/fr_train_data_pgc_v5_score_1020_30m.json"

prev =       "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_20260331_5000w_n_300w_high_5100w_shuffled.json"
new =        "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/va_train_data_pgc_v5_score_20260424_neg_300w.json"
merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/video/train_data_v5_score_va_20260424_54m.json"
concat_json_lists( prev, new, merged_new, shuffle_out=True, sort_out=False)

2000
200
shuffled
2200


In [6]:
with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_20260220_67w.pkl', 'rb') as fr:
# with open('/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_1216_1000w.pkl', 'rb') as fr:
    all_music_caption_dict = pickle.load(fr)
    print(f"Totally {len(all_music_caption_dict)} video music ids in meta info dict")

Totally 678663 video music ids in meta info dict


In [7]:
all_music_caption_dict

{100798787: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/100798787.json',
 11722: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/11722.json',
 127761614: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/127761614.json',
 142342892496379904: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/142342892496379904.json',
 145538627098468352: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/145538627098468352.json',
 172780247556173824: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/172780247556173824.json',
 222478451335811072: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/222478451335811072.json',
 222659167927533568: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/222659167927533568.json',
 222851382759124992: '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_67w_by_mid_20260220/22285138

In [11]:
all_music_caption_dict[7588574674037967632]

KeyError: 7588574674037967632

## Merge Music pkl

In [20]:
old_pkl = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_1216_1000w_plus_200w.pkl"
new_pkl = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_20260301_1w3.pkl"
new_pkl2 = '/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_20260223_135w.pkl'
merged_new = "/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_train_20260301_1336w.pkl"

In [16]:
with open(old_pkl,'rb') as f1, \
     open(new_pkl,'rb') as f2, \
     open(new_pkl2,'rb') as f3:
    d1 = pickle.load(f1)
    d2 = pickle.load(f2)
    d3 = pickle.load(f3)
print(len(d1), len(d2), len(d3))
merged = {**d1, **d2, **d3}  # second wins
print(len(merged), len(d1)+len(d2)+len(d3))
with open(merged_new,'wb') as out:
    pickle.dump(merged, out)
print("merged done")

12061413 12047 1355042
12219852 13428502
merged done


In [22]:
with open(merged_new, 'rb') as f:
    data = pickle.load(f)

In [25]:
data[7277920922448595717]

'/mnt/bn/jiny-ttls-i18n-fr1q/lutong/data/music/music_135w_by_mid_20260223/7277920922448595717.json'